# Creating a GeoZarr File
This notebook demonstrates how to convert data into the GeoZarr format. This module uses the python Zarr library to convert various data types into a GeoZarr compliant Zarr file. The library currently has the capability to convert both GeoTiff and NetCDF files.

In [10]:
%load_ext autoreload
%autoreload 2

from geo_zarr import GeoZarr, ZarrStorage
from helpers import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Creating a GeoZarr file using GeoTiffs:
In this first example, we will create a GeoZarr file using a set of GeoTiffs. The following assumptions are made regarding the GeoTiff files:
- Each GeoTiff describes the observation spatially at a given point in time. The timestamp is currently read from the filename.
- Each GeoTiff needs to have the same extent, resolution and projection. i.e. they should share the same Coordinate Reference System (CRS).

Let's get started and first define some file paths. We need to specify the following:
- The filepath pointing at where the observation data is stored.
- The filepath pointing at where the uncertainty data is stored. Use None if there is no associated uncertainties.
- The filepath pointing at where the masks are stored. Currently the masks are stored as multi-polygons in a GeoPandas DataFrame.
- The filepath where we want the final GeoZarr to be stored.

In [ ]:
folder_input_path = (
        "/data/yak1/working/polarice/polarice_project_outputs/"
        "timeseries_results_poca_correction/outputs/amundsen_swath_tide_corrected/medians_tifs"
    )

folder_error_path = (
        "/data/yak1/working/polarice/polarice_project_outputs/timeseries_results_poca_correction"
        "/outputs/amundsen_swath_tide_corrected/errors_tifs"
    )

file_path_mask = "/data/yak1/working/dtc_ice_sheets/masks/thwaites.gpkg"

folder_output_path = ("test_geozarr_method1.zarr"
    )

We can now get lists of all the file names for the observations and uncertainties using the provided <i>get_list_of_tiffs_from_folder()</i> helper function, which scans the specified directory and returns a list of all the GeoTiffs contained within it.

In [12]:
observation_tifs = get_list_of_tiffs_from_folder(folder_path=folder_input_path)
error_tifs = get_list_of_tiffs_from_folder(folder_path=folder_error_path)

We need to provide Climate and Forecast (CF) standard names and a standard reference date/time to convert timestamps to seconds. Specifically:
- <i>observation_names</i>: Is a dictionary mapping used to remap the column names in the intermediate pd.DataFrame to CF standard names.
- <i>coord_names</i>: A list of the final coordinate names. Importantly the order here corresponds to the final order of the axes in the datacube.
- <i>reference_time</i>: The reference time use to convert the timestamps to seconds from reference time. 

In [13]:
observation_names = {
        "value": "height_difference_from_reference_elevation",
        "x": "projection_x_coordinate",
        "y": "projection_y_coordinate",
    }
coord_names = ["projection_x_coordinate", "projection_y_coordinate", "time"]
reference_time = pd.Timestamp(year=1970, month=1, day=1)

Let's now create the GeoZarr object. This will currently be empty, as we will add the data shortly. 

It is possible to create the GeoZarr object either in memory, or stored on the local disk. In this case we will choose the local disk option, where we need to provide a file path for where we want the GeoZarr file to be stored. It is also necessary to specify other information which agnostic as to whether the GeoZarr is stored in memory or on disk:
- <i>zarr_format</i>: The available options are either 2 or 3. Currently this is limited to Zarr 2, due to how recently Zarr 3 was released.
- <i>n_chunks</i>: This is the number of chunks any data arrays within the GeoZarr will be chunked into. Note when chunking, this is the number of chunks along each axis!

In [14]:
geozarr = GeoZarr(
        storage_type=ZarrStorage.local_store, storage_directory=folder_output_path, zarr_format=2, n_chunks=5
    )

The last thing that we need to define is the CRS for the GeoZarr. 

Firstly we grab the geotransform in the GDAL raster data model form by extracting it directly from the GeoTiff file corresponding to the first observation, but can also be found using the extent and resolution. We then create the CRS using the <i>create_coordinate_reference_system()</i> method. There are a number of optional parameters which can be passed, please see the method docstring for more information. Importantly, we set the <i>add_attributes</i> parameter to True so that the CRS is stored directly in the root GeoZarr attribute.

In [15]:
data_tiff = RasterNp.load_from_tiff(path.join(folder_input_path, observation_tifs[0]))

geotransform = get_gdal_geotransform(
    data_tiff._get_transform(
        bounding_box=data_tiff.bounding_box,
        resolution=data_tiff.resolution,
        y_affine_negative=data_tiff.y_affine_negative,
    )
)

crs = geozarr.create_coordinate_reference_system(
        grid_mapping_name="polar_stereographic",
        geotransform=geotransform,
        srid="urn:ogc:def:crs:EPSG::3031",
        crs_wkt='PROJCS["WGS 84 / Antarctic Polar Stereographic",'
        'GEOGCS["WGS 84",DATUM["WGS_1984",'
        'SPHEROID["WGS 84",6378137,298.257223563,'
        'AUTHORITY["EPSG","7030"]],'
        'AUTHORITY["EPSG","6326"]],'
        'PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],'
        'UNIT["degree",0.0174532925199433,'
        'AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],'
        'PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-71],'
        'PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],'
        'PARAMETER["false_easting",0],'
        'PARAMETER["false_northing",0],'
        'UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],'
        'AXIS["Northing",NORTH],AUTHORITY["EPSG","3031"]]',
        add_attributes=True,
    )

[ WARNING 2025-05-28 BST 09:34:41.633 pid2175973,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_sheets/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


We can now populate the GeoZarr file with the data. This is achieved in two steps, and there are two methods of doing this which differ based on the intermediate processing steps chosen.
1. The GeoTiff files are read directly into a datacube, which is then saved as the GeoZarr file.
2. The GeoTiff files are first read in as a pd.DataFrame. This DataFrame is then converted to a datacube and written into the GeoZarr file. 

### Converting GeoTiff to GeoZarr - Method 1 via a Datacube
First let's look at the direct conversion of the GeoTiff files into a datacube without first converting to a pandas DataFrame object. The first step is to load the GeoTiff files into a datacube. For this the <i>load_tiff2datacube</i> function is used. 

There are also a number of parameters here which can be used to control the way the data is processed before being written into the GeoZarr:
- Filepaths to where the observation and error GeoTiff are stored. If both a file path for observations and uncertainties, then it is expected that the respective GeoTiffs will share the same file names.
- The filepath to where the mask is stored, and a list of the associated polygon to be used in the mask.
- Parameters which specify how the datetimes should be processed.

In [16]:
data_observations, data_errors, data_coords, mask_raster = load_tiff2datacube(
    file_names=observation_tifs,
    folder_path_observations=folder_input_path,
    reference_time=reference_time,
    folder_path_errors=folder_error_path,
    file_path_mask=file_path_mask,
    create_raster_mask=True,
    geometry_name="all",
    process_datetimes=True,
    process_filenames_as_dates="dates",
)

  0%|          | 0/132 [00:00<?, ?it/s]

[ WARNING 2025-05-28 BST 09:34:43.093 pid2175973,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_sheets/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.
[ WARNING 2025-05-28 BST 09:34:43.373 pid2175973,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_sheets/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.
  6%|▌         | 8/132 [00:01<00:15,  7.76it/s][ WARNING 2025-05-28 BST 09:34:44.631 pid2175973,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_she

The second step is to save the datacube into the GeoZarr file using the <i>datacube2geozarr()</i> method. This method writes the datacube into the GeoZarr file as variables with the associated coordinates and metadata. In this case two variables are being added as both the measured height differences and the associated measurement uncertainties.

In [8]:
geozarr.datacube2geozarr(
        data_cube=data_observations,
        data_columns=[
            "height_difference_from_reference_elevation",
        ],
        data_coords=data_coords,
        coordinate_columns=coord_names,
        auxiliary_columns=[],
    )

geozarr.datacube2geozarr(
        data_cube=data_observations,
        data_columns=[
            "height_difference_from_reference_elevation_uncertainty",
        ],
        data_coords=data_coords,
        coordinate_columns=coord_names,
        auxiliary_columns=[],
    )

In this case we will also save a copy of the rasterised mask to the GeoZarr file. This is saved as a variable and as specified by the GeoZarr specification must use the same CRS as the other data stored in the GeoZarr.

In [9]:
data_coord_y, data_coord_x = calculate_coords_from_geotransform(
        geotransform=geotransform, x_dim=data_tiff.shape[1], y_dim=data_tiff.shape[0]
    )

geozarr.add_raster(
    raster_data=mask_raster,
    data_coords={
        "projection_y_coordinate": data_coord_y,
        "projection_x_coordinate": data_coord_x,
    },
    coord_names=["projection_y_coordinate", "projection_x_coordinate"],
    attributes={
        "standard_name": "mask",
        "_ARRAY_DIMENSIONS": ["projection_y_coordinate", "projection_x_coordinate"],
    },
    name="mask",
)

### Converting GeoTiff to GeoZarr - Method 2 via a DataFrame

We can now repeat the exercise but instead this time using the processing chain which converts first to a DataFrame. It is recommended to use the previous method (Method 1), unless there is an explicit need to first convert to a DataFrame first.

In [ ]:
folder_output_path = (
        "test_geozarr_method2.zarr"
    )

geozarr = GeoZarr(
        storage_type=ZarrStorage.local_store, storage_directory=folder_output_path, zarr_format=2, n_chunks=5
    )

data, mask_raster = load_tiffs2dataframe(
        file_names=observation_tifs,
        folder_path_observations=folder_input_path,
        observation_names=observation_names,
        coord_names=coord_names,
        reference_time=reference_time,
        error_file_names=error_tifs,
        folder_path_errors=folder_error_path,
        file_path_mask=file_path_mask,
        create_raster_mask=True,
        geometry_name="all",
        process_datetimes=True,
        process_filenames_as_dates="dates",
    )

[ WARNING 2025-05-19 BST 16:14:51.042 pid1889931,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_sheets/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.
[ WARNING 2025-05-19 BST 16:14:51.147 pid1889931,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_sheets/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.
[ WARNING 2025-05-19 BST 16:14:51.575 pid1889931,                    __init__.py, ln 356, in                      open()] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/palmerjohn/.conda/envs/dtc_ice_sheets/share/proj/proj.db contains DATABASE.LAYOUT

In [11]:
geozarr.dataframe2geozarr(
        data=data,
        data_columns=[
            "height_difference_from_reference_elevation",
            "height_difference_from_reference_elevation_uncertainty",
        ],
        coordinate_columns=["time", "projection_y_coordinate", "projection_x_coordinate"],
        auxiliary_columns=[],
        y_affine_negative=True,
        uncenter_cols=["projection_y_coordinate", "projection_x_coordinate"],
        x_resolution=float(geotransform.split(" ")[1]),
        y_resolution=float(geotransform.split(" ")[5]),
    )

Finally it is currently recommended to store any masks used within the GeoZarr to provide traceability for previous masking. In the previous step the rasterised mask is returned from the <i>load_tiffs2dataframe()</i> method. This can then be stored within the GeoZarr using the add_raster method as long as it is on the same coordinates as the other spatial data stored within the GeoZarr.

We can define the coordinates of the mask with the geotransform using the extent and the resolution. Note that the intermediate processing of converting to a DataFrame results in the loss of one row and column of data through the centering and then the subsequent recentering.

In [12]:
data_coord_y, data_coord_x = calculate_coords_from_geotransform(
        geotransform=geotransform, x_dim=data_tiff.shape[1], y_dim=data_tiff.shape[0]
    )

geozarr.add_raster(
    raster_data=mask_raster,
    data_coords={
        "projection_y_coordinate": data_coord_y,
        "projection_x_coordinate": data_coord_x,
    },
    coord_names=["projection_y_coordinate", "projection_x_coordinate"],
    attributes={
        "standard_name": "mask",
        "_ARRAY_DIMENSIONS": ["projection_y_coordinate", "projection_x_coordinate"],
    },
    name="mask",
)

### Creating a GeoZarr from a NetCDF

Here we create a GeoZarr directly from a NetCDF file. This is generally a much simpler process than when loading from a set of GeoTiff files.

Once again we need to specify the file paths to the NetCDF file and the location where we would like the GeoZarr to be stored.

In [ ]:
folder_input_path = (
        "/data/yak1/working/dtc_ice_sheets/data/Paolo 3D Cube of Basel Melt/NSIDC-0792_19920317-20171216_V01.0.nc"
    )
folder_output_path = "test_geozarr_netcdf.zarr"

Let's create the GeoZarr, this is done in the same way as when loading data from GeoTiffs in the example above.

In [14]:
geozarr = GeoZarr(
        storage_type=ZarrStorage.local_store, storage_directory=folder_output_path, zarr_format=2, n_chunks=5
    )

We need to specify the variables to process in the NetCDF file as well as the coordinate names. In this example I haven't bothered converting the variable names to standard CF names, although the coordinates variables will be converted automatically as long as they are one of "x", "y", "X", "Y" or "time".

In [15]:
target_columns = [
        "melt",
        "melt_mean",
        "melt_err",
        "height_change",
        "height_change_err",
        "fac",
        "fac_mean",
        "fac_err",
        "smb",
        "smb_mean",
        "smb_err",
        "thickness",
        "thickness_mean",
        "thickness_err",
        "ID",
    ]

coordinate_columns = ["time", "x", "y"]

Again we need to specify the CRS. In this case it is copied directly from the net CDF file explicately, but equally this could be created in the same way as before.

In [16]:
crs = geozarr.create_coordinate_reference_system(
    grid_mapping_name="polar_stereographic",
    straight_vertical_longitude_from_pole=0,
    latitude_of_projection_origin=-90,
    false_easting=0,
    false_northing=0,
    semi_major_axis=6378137.0,
    inverse_flattening=298.257223563,
    geotransform="-2799367.5 1920.0 0 2799367.5 0 -1920.0",
    srid="urn:ogc:def:crs:EPSG::3031",
    proj4text="proj4text",
    standard_parallel="standard_parallel",
    crs_wkt='PROJCS["WGS 84 / Antarctic Polar Stereographic",'
    'GEOGCS["WGS 84",DATUM["WGS_1984",'
    'SPHEROID["WGS 84",6378137,298.257223563,'
    'AUTHORITY["EPSG","7030"]],'
    'AUTHORITY["EPSG","6326"]],'
    'PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],'
    'UNIT["degree",0.0174532925199433,'
    'AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],'
    'PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-71],'
    'PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],'
    'PARAMETER["false_easting",0],'
    'PARAMETER["false_northing",0],'
    'UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],'
    'AXIS["Northing",NORTH],AUTHORITY["EPSG","3031"]]',
    add_attributes=True,
)

The NetCDF can now be converted to a GeoZarr using the <i>convert_netcdf_to_geo_zarr()</i> method.

In [7]:
geozarr.convert_netcdf_to_geo_zarr(
        filepath=folder_input_path, target_columns=target_columns, coordinate_columns=coordinate_columns
    )

NameError: name 'folder_input_path' is not defined